# Audio Event Detection — Data Download & Setup (Colab)

This notebook downloads and prepares all training data, then saves it permanently to Google Drive.

**Run this once.** After completion, your Drive will contain all raw audio files ready for training in a separate session.

**Datasets used:**
| Dataset | Size | Classes covered | Download |
|---|---|---|---|
| ESC-50 | ~600 MB | ~30 / 47 | Auto |
| FSD50K | ~30 GB | ~42 / 47 | Auto |
| UrbanSound8K | ~6 GB | ~9 / 47 | Manual |

> You can run the cells for whichever datasets you want. At minimum, run **ESC-50** to get started quickly.

## 1. Mount Google Drive & Clone Repo

In [ ]:
# Mount Google Drive — all data will be saved here permanently
from google.colab import drive
drive.mount('/content/drive')

import os

# Paths
DRIVE_DIR = '/content/drive/MyDrive/audio-event-detection'
DRIVE_DATA_DIR = os.path.join(DRIVE_DIR, 'data')
DRIVE_RAW_DIR = os.path.join(DRIVE_DATA_DIR, 'raw')
PROJECT_DIR = '/content/audio-event-detection'

# Create Drive directories
for d in [DRIVE_DIR, DRIVE_DATA_DIR, DRIVE_RAW_DIR,
          os.path.join(DRIVE_DIR, 'checkpoints'), os.path.join(DRIVE_DIR, 'logs')]:
    os.makedirs(d, exist_ok=True)

print(f"Drive data directory: {DRIVE_DATA_DIR}")
print("✓ Drive mounted and directories created")

In [ ]:
# Clone the repository
import subprocess

if os.path.exists(PROJECT_DIR):
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    # TODO: Replace with your actual repo URL
    REPO_URL = "https://github.com/YOUR_USERNAME/audio-event-detection.git"
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

# Add project to Python path
import sys
sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print("✓ Repo ready")

In [ ]:
# Install dependencies
!pip install -q torch torchaudio timm librosa soundfile scikit-learn pyyaml matplotlib tqdm
print("✓ Dependencies installed")

## 2. Download ESC-50 (~600 MB, ~30 classes)
This is the smallest dataset and the quickest way to get started.

In [ ]:
# Download ESC-50
!python scripts/download_esc50.py \
    --output-dir data/downloads/esc50 \
    --labels-csv data/downloads/esc50_labels.csv

print("\n✓ ESC-50 download and label preparation complete")

## 3. Download FSD50K (~30 GB, ~45 classes) [OPTIONAL — best coverage]

FSD50K is large but provides the best class coverage. You can:
- Download **metadata only** first (tiny) to check label coverage
- Download **dev set only** (~22 GB) for most of the data
- Download **everything** (~30 GB) for full coverage

Skip this section if you only want ESC-50 for now.

In [ ]:
# Option A: Metadata only (check coverage without downloading audio)
# !python scripts/download_fsd50k.py --metadata-only \
#     --output-dir data/downloads/fsd50k \
#     --labels-csv data/downloads/fsd50k_labels.csv

# Option B: Dev set only (~22 GB) — recommended
!python scripts/download_fsd50k.py --dev-only \
    --output-dir data/downloads/fsd50k \
    --labels-csv data/downloads/fsd50k_labels.csv

# Option C: Full dataset (~30 GB)
# !python scripts/download_fsd50k.py \
#     --output-dir data/downloads/fsd50k \
#     --labels-csv data/downloads/fsd50k_labels.csv

print("\n✓ FSD50K download complete")

## 4. Prepare UrbanSound8K [OPTIONAL — manual download]

UrbanSound8K requires manual download due to license terms.

**Steps:**
1. Visit https://urbansounddataset.weebly.com/urbansound8k.html
2. Download the archive
3. Upload & extract to `data/downloads/urbansound8k/UrbanSound8K/` (or upload to Drive and copy)
4. Run the cell below

In [ ]:
# Uncomment after you've placed UrbanSound8K files in the expected location

# # If you uploaded to Drive:
# !cp -r "/content/drive/MyDrive/UrbanSound8K" data/downloads/urbansound8k/UrbanSound8K

# !python scripts/download_urbansound8k.py \
#     --input-dir data/downloads/urbansound8k/UrbanSound8K \
#     --labels-csv data/downloads/urbansound8k_labels.csv

# print("\n✓ UrbanSound8K preparation complete")

## 5. Merge Datasets & Copy Audio

This step merges all downloaded datasets into a single unified `labels.csv` and copies/resamples all audio files to 16kHz mono WAV format.

In [ ]:
# Merge all available datasets into unified format
# Audio is resampled to 16kHz mono and copied to data/raw/
!python scripts/prepare_multi_dataset.py --data-dir data --sample-rate 16000

print("\n✓ Dataset merge complete")

In [ ]:
# Check the class distribution
import json

stats_path = "data/class_statistics.json"
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)

    print(f"Total clips: {stats['total_clips']}")
    print(f"Active classes: {stats['num_classes']} / 47")
    print(f"\nSources: {stats['sources']}")

    if stats.get('missing_classes'):
        print(f"\n⚠ Missing classes ({len(stats['missing_classes'])}):")
        for cls in stats['missing_classes']:
            print(f"  - {cls}")

    print(f"\nTop 10 classes:")
    for cls, count in list(stats['class_counts'].items())[:10]:
        bar = "█" * min(40, count // 20)
        print(f"  {cls:25s} {count:5d} {bar}")
else:
    print("No statistics file found. Run the merge step first.")

In [ ]:
# Verify the outputs
import glob

n_raw = len(glob.glob("data/raw/*.wav"))
has_labels = os.path.exists("data/labels.csv")
has_classmap = os.path.exists("data/class_map.json")

print(f"Raw audio files:  {n_raw}")
print(f"labels.csv:       {'✓' if has_labels else '✗'}")
print(f"class_map.json:   {'✓' if has_classmap else '✗'}")

## 6. Save Everything to Google Drive

This copies the raw audio and label files to Drive so they persist across Colab sessions.
Spectrograms will be extracted separately in a later step.

In [ ]:
import shutil

print("Copying prepared data to Google Drive...")
print("This may take a while for large datasets.\n")

# Files to copy to Drive data dir
files_to_copy = [
    "data/labels.csv",
    "data/class_map.json",
    "data/class_statistics.json",
]

for f in files_to_copy:
    if os.path.exists(f):
        dest = os.path.join(DRIVE_DATA_DIR, os.path.basename(f))
        shutil.copy2(f, dest)
        print(f"  ✓ {f} → Drive")

# Copy raw audio
print("\nCopying raw audio to Drive (this may take a while)...")
if os.path.isdir("data/raw"):
    raw_files = glob.glob("data/raw/*.wav")
    for i, src in enumerate(raw_files):
        dest = os.path.join(DRIVE_RAW_DIR, os.path.basename(src))
        if not os.path.exists(dest):
            shutil.copy2(src, dest)
        if (i + 1) % 500 == 0:
            print(f"  Progress: {i + 1}/{len(raw_files)}")
    print(f"  ✓ {len(raw_files)} audio files copied")

print("\n" + "=" * 60)
print("✓ ALL DATA SAVED TO GOOGLE DRIVE")
print(f"  Location: {DRIVE_DATA_DIR}")
print("=" * 60)

In [ ]:
# Final verification: check Drive contents
drive_raw = len(glob.glob(os.path.join(DRIVE_RAW_DIR, "*.wav")))
drive_labels = os.path.exists(os.path.join(DRIVE_DATA_DIR, "labels.csv"))
drive_classmap = os.path.exists(os.path.join(DRIVE_DATA_DIR, "class_map.json"))

print("=== Google Drive Contents ===")
print(f"  Raw audio:      {drive_raw} files")
print(f"  labels.csv:     {'✓' if drive_labels else '✗'}")
print(f"  class_map.json: {'✓' if drive_classmap else '✗'}")
print()

if drive_raw > 0 and drive_labels and drive_classmap:
    print("✓ Data setup complete! You can now close this notebook")
    print("  and open the training notebook in a new Colab session.")
    print(f"\n  Your data is permanently saved at:")
    print(f"  {DRIVE_DATA_DIR}")
else:
    print("⚠ Some files are missing. Check the steps above for errors.")

## 7. Extract Spectrograms & Save to Drive

Convert all raw audio files to **log-mel spectrograms** (128 mel bands, 5s clips) and save as `.npy` files on Drive.
This pre-computes features so training loads instantly without on-the-fly extraction.

In [ ]:
import numpy as np
import librosa
import glob
import os
from tqdm import tqdm

# --- Paths ---
DRIVE_RAW_DIR = "/content/drive/MyDrive/audio-event-detection/data/raw"
DRIVE_SPEC_DIR = "/content/drive/MyDrive/audio-event-detection/data/spectrograms"
os.makedirs(DRIVE_SPEC_DIR, exist_ok=True)

# --- Spectrogram parameters (match configs/default.yaml) ---
SAMPLE_RATE = 16000
CLIP_DURATION = 5.0          # seconds
N_MELS = 128
WINDOW_SIZE_MS = 25.0
HOP_LENGTH_MS = 10.0
F_MIN = 50.0
F_MAX = 8000.0
LOG_OFFSET = 1e-6

TARGET_LENGTH = int(SAMPLE_RATE * CLIP_DURATION)  # 80000 samples
N_FFT = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000.0)  # 400
HOP_LENGTH = int(SAMPLE_RATE * HOP_LENGTH_MS / 1000.0)  # 160

def preprocess_and_extract(file_path):
    """Load audio, normalize, pad/trim, compute log-mel spectrogram."""
    # Load & resample to 16 kHz mono
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

    # Peak-normalize
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    # Pad or trim to exactly 5 seconds
    if len(audio) > TARGET_LENGTH:
        start = (len(audio) - TARGET_LENGTH) // 2
        audio = audio[start : start + TARGET_LENGTH]
    elif len(audio) < TARGET_LENGTH:
        pad_total = TARGET_LENGTH - len(audio)
        audio = np.pad(audio, (pad_total // 2, pad_total - pad_total // 2))

    # Compute log-mel spectrogram → shape (128, 501)
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, power=2.0,
    )
    log_mel = np.log(mel_spec + LOG_OFFSET).astype(np.float32)
    return log_mel

# --- Find all raw audio files ---
audio_files = sorted(glob.glob(os.path.join(DRIVE_RAW_DIR, "*.wav")))
print(f"Found {len(audio_files)} raw audio files in Drive")

# --- Extract spectrograms (skip already-processed files) ---
done, skipped, errors = 0, 0, 0
for fpath in tqdm(audio_files, desc="Extracting spectrograms"):
    stem = os.path.splitext(os.path.basename(fpath))[0]
    save_path = os.path.join(DRIVE_SPEC_DIR, f"{stem}.npy")

    if os.path.exists(save_path):
        skipped += 1
        continue

    try:
        spec = preprocess_and_extract(fpath)
        np.save(save_path, spec)
        done += 1
    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f"  ⚠ Failed: {os.path.basename(fpath)} — {e}")

print(f"\n{'=' * 60}")
print(f"✓ Spectrogram extraction complete")
print(f"  New:     {done}")
print(f"  Skipped: {skipped} (already exist)")
print(f"  Errors:  {errors}")
print(f"  Output:  {DRIVE_SPEC_DIR}")
print(f"{'=' * 60}")

In [ ]:
# Verify spectrograms
import glob, os, numpy as np

DRIVE_SPEC_DIR = "/content/drive/MyDrive/audio-event-detection/data/spectrograms"
spec_files = sorted(glob.glob(os.path.join(DRIVE_SPEC_DIR, "*.npy")))
print(f"Spectrograms on Drive: {len(spec_files)}")

if spec_files:
    # Check a sample spectrogram
    sample = np.load(spec_files[0])
    print(f"Sample shape: {sample.shape}  (expected: 128 × 501)")
    print(f"dtype: {sample.dtype}, min: {sample.min():.2f}, max: {sample.max():.2f}")

    # Quick sanity: all should have the same shape
    bad = 0
    for f in spec_files[:100]:
        s = np.load(f)
        if s.shape != sample.shape:
            bad += 1
            print(f"  ⚠ Shape mismatch: {os.path.basename(f)} → {s.shape}")
    if bad == 0:
        print("✓ All checked spectrograms have consistent shape")
else:
    print("⚠ No spectrograms found. Run the extraction cell above first.")

## Next Steps

Data and spectrograms are now permanently saved to Google Drive. In a **new Colab session**:

1. Open the **training notebook** (`notebooks/colab_training.ipynb`)
2. It will load spectrograms directly from Drive — no re-downloading or re-processing needed
3. Train/val/test splits and training will be handled in the training notebook
4. Training checkpoints also save to Drive, so you can resume across sessions

```
Drive structure after this notebook:
/MyDrive/audio-event-detection/
├── data/
│   ├── raw/              ← audio files (16kHz mono WAV)
│   ├── spectrograms/     ← log-mel spectrograms (.npy)
│   ├── labels.csv        ← unified multi-label annotations
│   ├── class_map.json    ← class name → index mapping
│   └── class_statistics.json
├── checkpoints/          ← (created during training)
└── logs/                 ← (created during training)
```